# Fink/LSST — Statistics of Alerts on Heatmap

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

- **author** : Sylvie Dagoret-Campane
- **affliliation** : IJCLab/CNRS
- **creation date** : 2026-04-02
- **last update** : 2026-04-03
- **subject:** Fink alert Broker applied to Rubin LSST alertes

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_05"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per group
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ────────────────────────────────────────────────────────
# Choose which groups to display:
#   'all'     -> all groups found on disk (default)
#   'calib'   -> only groups usable for photometric calibration
#   'exclude' -> only groups EXCLUDED from calibration
#                (variable stars, SSO objects, TNS transients, blazars...)
PLOT_MODE = "all"  # <── change here: 'all' | 'calib' | 'exclude'

# Groups excluded from calibration (consistent with notebook 01).
# Any group whose name STARTS WITH 'simbad_variable' is also excluded.
CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "gaia_star_variable",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (identical to notebook 01 + luptitude support)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties.

    Negative or zero flux values are silently returned as NaN.

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.
    flux_err_nJy : array-like or None
        1-sigma flux uncertainty in nJy.  If None, only magnitudes are returned.

    Returns
    -------
    mag : ndarray
        AB magnitude (NaN where flux <= 0).
    mag_err : ndarray or None
        Magnitude uncertainty (NaN where flux <= 0), or None if not requested.
    """
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes).

    Luptitudes (Lupton et al. 1999) handle zero and negative flux values
    that arise in difference-image photometry (DIA).  They behave as
    standard AB magnitudes at high signal-to-noise and transition to a
    linear flux scale near the noise floor, so every measurement is
    plotted -- including those with negative flux.

    The softening parameter *b* is set to the median flux uncertainty of
    the input array, placing the linear/log transition at the 1-sigma
    noise level.  This matches the convention used in the DP2 DiaObject
    Butler notebook (``plot_dia_object_luptitudes``).

    Formula
    -------
    mu = -2.5 * log10(e) * [arcsinh(f / 2b) + ln(b)] + (zp + 2.5*log10(b))

    Error propagation
    -----------------
    sigma_mu = 1.085736 * sigma_f / sqrt(f^2 + (2b)^2)

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.  May contain negative values.
    flux_err_nJy : array-like
        1-sigma flux uncertainty in nJy (used to derive the softening
        parameter b = median(flux_err_nJy)).
    zero_point : float, optional
        AB zero-point offset in nJy units (default 31.4 for nJy).

    Returns
    -------
    lup : ndarray
        Asinh magnitude (Luptitude) for each measurement.
    lup_err : ndarray
        1-sigma uncertainty on the Luptitude.
    b : float
        Softening parameter used (= median of flux_err_nJy).
    """
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)

    # Softening parameter: median noise level of the input data
    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0  # safe fallback

    # Pre-factor  2.5 / ln(10) ~ 1.085736
    log10_e_inv = 1.085736

    # Asinh magnitude (Luptitude)
    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))

    # Error propagation
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)

    return lup, lup_err, b


print("flux_to_luptitude() defined.")

In [ ]:
def plot_focal_plane_heatmap_vectorized(
    alerts_df,
    geometry_csv,
    value_col="r:visit",
    agg_func="count",
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
):

    # --- Load geometry
    geom = pd.read_csv(geometry_csv)

    # --- Compute values per detector
    values = alerts_df.groupby("r:detector")[value_col].agg(agg_func)
    geom["value"] = geom["detector"].map(values).fillna(0).astype(float)

    # Must compute boundaries
    xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()

    # --- Handle log scale and normalization
    if log_scale:
        vmin = max(geom["value"][geom["value"] > 0].min(), 1e-2)
        vmax = geom["value"].max()
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = mcolors.Normalize(vmin=geom["value"].min(), vmax=geom["value"].max())

    cmap = plt.get_cmap(cmap)

    # --- Plot
    fig, ax = plt.subplots(figsize=figsize)

    # Vectorisé : on crée toutes les patches en une fois
    for i, row in geom.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row["value"]
        color = cmap(norm(val)) if val > 0 else "lightgrey"
        poly = patches.Polygon(corners, facecolor=color, edgecolor="black", linewidth=0.2)
        ax.add_patch(poly)
        ax.text(
            row["x_center"],
            row["y_center"],
            f"{int(row['detector'])}\n{row['name']}",
            ha="center",
            va="center",
            fontsize=6,
            fontweight="bold",
        )

    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")
    ax.set_title(title)

    # --- Colorbar
    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

## 3. Load Parquet files from disk

In [ ]:
# Rubin LSST Focal Plane Geometry

In [ ]:
PATH_FOCALPLANEGEOM_filename = "data_FocalPlane/ccd_geometry.csv"
geometry_csv = PATH_FOCALPLANEGEOM_filename

In [ ]:
# Auto-discover available groups from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))


# ── Apply PLOT_MODE filter ───────────────────────────────────────────────
def _is_excluded(g):
    return g in CALIBRATION_EXCLUDE or g.startswith("simbad_variable")


if PLOT_MODE == "calib":
    selected_groups = [g for g in all_groups if not _is_excluded(g)]
elif PLOT_MODE == "exclude":
    selected_groups = [g for g in all_groups if _is_excluded(g)]
else:  # 'all'
    selected_groups = list(all_groups)

print(f"Groups on disk: {len(all_groups)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_groups)}")
for g in all_groups:
    has_fp = "yes" if g in groups_fp else "no"
    has_src = "yes" if g in groups_src else "no"
    sel = "<<" if g in selected_groups else "  "
    label = "[EXCL] " if _is_excluded(g) else "[calib]"
    print(f"  {sel} {label} {g:40s}  fp={has_fp:3s}  src={has_src}")

In [ ]:
# Load all Parquet files and reconstruct the lc_cache dictionary.
# Structure: lc_cache[group][oid] = {'fp': DataFrame, 'src': DataFrame}

lc_cache = {}

for group in all_groups:
    lc_cache[group] = {}

    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])

        # Drop any residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag / mag_err if absent (e.g. columns were not saved)
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # Split by object
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

# Summary
print("Objects loaded per group:")
for g in all_groups:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:35s}  {n_oids:3d} objects  {n_pts:6d} data points")

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['fp']

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['src']

## Prepare heatmap

In [ ]:
def concat_group_src(lc_cache, group="gaia_star_stable"):
    dfs = []

    for diaObjectId, content in lc_cache[group].items():
        if "src" in content and content["src"] is not None:
            df = content["src"].copy()

            # garder la trace de l'origine (très important)
            df["diaObjectId"] = diaObjectId

            dfs.append(df)

    if len(dfs) == 0:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

In [ ]:
df_alerts_gaia_star_stable = concat_group_src(lc_cache, "gaia_star_stable")
df_alerts_gaia_star_stable_hq = concat_group_src(lc_cache, "gaia_star_stable_hq")

In [ ]:
df_all_alerts_gaia_star = pd.concat([df_alerts_gaia_star_stable, df_alerts_gaia_star_stable_hq])

In [ ]:
df_all_alerts_gaia_star.groupby("r:detector")["r:visit"].count().describe()

## Plot Focal Plane Heatmap

In [ ]:
plot_focal_plane_heatmap_vectorized(
    df_all_alerts_gaia_star,
    geometry_csv,
    value_col="r:visit",
    agg_func="count",
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Alert counts on LSSTCam Focal Plane Heatmap",
)

## Radial projection: alert count vs. distance from camera centre

Each CCD is projected onto a single radial coordinate
$r = \sqrt{x_{\rm center}^2 + y_{\rm center}^2}$ [mm],
where $(x_{\rm center},\, y_{\rm center})$ are the CCD centre coordinates
in the focal-plane frame (origin = optical axis of the camera).

### Three competing models

| Model | Formula | Parameters | Physical motivation |
|---|---|---|---|
| M1 — Cosine | $N = A\cos(Br)$ | 2 | solid-angle projection, first order |
| M2 — Cosine² | $N = A\cos^2(Br)$ | 2 | intensity ∝ cos² (Malus-like) |
| M3 — Full | $N = N_0\cos(Kr)^\beta\left(1+r^2/r_0^2\right)^{-\gamma}$ | 5 | projection + vignetting / field illumination drop |

All three fits are restricted to the inner field $r \leq R_{\rm fit}$ (default 100 mm).  
The fitted curves are extended analytically beyond $R_{\rm fit}$ as dash-dot lines.

### Uncertainties

Alert counts follow approximately Poisson statistics, so the 1-sigma uncertainty
on each CCD count is $\sigma_N = \sqrt{N}$.  These are used as weights in all fits.

In [ ]:
# ── Build the per-CCD alert count table with radial distance ─────────────────

# Load geometry
geom = pd.read_csv(geometry_csv)

# Compute radial distance of each CCD centre from the camera optical axis
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# Count alerts per detector (same aggregation as the heatmap)
alert_counts_per_ccd = (
    df_all_alerts_gaia_star.groupby("r:detector")["r:visit"]
    .count()
    .rename("n_alerts")
    .reset_index()
    .rename(columns={"r:detector": "detector"})
)

# Merge geometry with alert counts
# CCDs with zero alerts are kept (n_alerts = 0) for completeness
geom_counts = geom.merge(alert_counts_per_ccd, on="detector", how="left")
geom_counts["n_alerts"] = geom_counts["n_alerts"].fillna(0).astype(float)

# Poisson error: sqrt(N); set to 1 for CCDs with 0 alerts to avoid division by zero
geom_counts["n_err"] = np.where(
    geom_counts["n_alerts"] > 0,
    np.sqrt(geom_counts["n_alerts"]),
    1.0,
)

# Keep only CCDs with at least one alert
geom_fit_all = geom_counts[geom_counts["n_alerts"] > 0].copy()

print(f"Total CCDs in geometry : {len(geom_counts)}")
print(f"CCDs with alerts       : {len(geom_fit_all)}")
print(f"r_mm range             : [{geom_fit_all['r_mm'].min():.1f},  {geom_fit_all['r_mm'].max():.1f}] mm")
print(
    f"N_alerts range         : [{geom_fit_all['n_alerts'].min():.0f},  {geom_fit_all['n_alerts'].max():.0f}]"
)

In [ ]:
# ── Model definitions and fits — restricted to r <= R_FIT_MAX ────────────────

# Maximum radius included in the fits.
# CCDs beyond R_FIT_MAX are displayed as open markers and described by
# the analytical extrapolation (dash-dot lines).
R_FIT_MAX = 120.0  # mm  <── change here if needed
R_FITPLOT_MAX = 150.0  # mm

# ── Model definitions ────────────────────────────────────────────────────────


def model_cos(r, A, B):
    """
    M1 — Cosine model: N(r) = A * cos(B * r)

    Parameters
    ----------
    r : array-like  — radial distance [mm]
    A : float       — on-axis normalisation
    B : float       — angular scale factor [rad/mm] = 1 / f_eff
    """
    return A * np.cos(B * np.asarray(r, dtype=float))


def model_cos2(r, A, B):
    """
    M2 — Cosine-squared model: N(r) = A * cos^2(B * r)

    Compared to M1, each photon's contribution is further weighted by
    another cos factor (Malus-like intensity projection).

    Parameters
    ----------
    r : array-like  — radial distance [mm]
    A : float       — on-axis normalisation
    B : float       — angular scale factor [rad/mm]
    """
    return A * np.cos(B * np.asarray(r, dtype=float)) ** 2


def model_full(r, N0, K, beta, r0, gamma):
    """
    M3 — Full 5-parameter model:

        N(r) = N0 * cos(K*r)^beta * (1 + (r/r0)^2)^(-gamma)

    The first factor is a generalised cosine projection with exponent beta
    (beta=1 recovers M1, beta=2 recovers M2 when K=B).
    The second factor is a Lorentzian-like vignetting / field-illumination
    envelope with scale radius r0 and slope gamma.

    Parameters
    ----------
    r     : array-like  — radial distance [mm]
    N0    : float       — on-axis normalisation
    K     : float       — angular scale factor [rad/mm]
    beta  : float       — cosine exponent (>0)
    r0    : float       — vignetting scale radius [mm] (>0)
    gamma : float       — vignetting power-law slope (>0)
    """
    r = np.asarray(r, dtype=float)
    cos_term = np.cos(K * r)
    # Guard against negative cosine values for large r (would produce NaN for
    # non-integer beta); clip to zero — physically no negative counts.
    cos_term = np.clip(cos_term, 0.0, None)
    lorentz_term = (1.0 + (r / r0) ** 2) ** (-gamma)
    return N0 * cos_term**beta * lorentz_term


# ── Split data into fit range and outer display-only range ───────────────────
mask_inner = geom_fit_all["r_mm"] <= R_FIT_MAX
mask_outer = geom_fit_all["r_mm"] > R_FIT_MAX

geom_inner = geom_fit_all[mask_inner].copy()  # used for all three fits
geom_outer = geom_fit_all[mask_outer].copy()  # shown but excluded from fits

r_data = geom_inner["r_mm"].values
N_data = geom_inner["n_alerts"].values
N_err = geom_inner["n_err"].values
N_max = N_data.max()

print(f"Fit range  : r <= {R_FIT_MAX:.0f} mm  ->  {len(geom_inner)} CCDs used for fits")
print(f"Outer range: r >  {R_FIT_MAX:.0f} mm  ->  {len(geom_outer)} CCDs shown (extrapolation only)")


def run_fit(model_func, p0, bounds, n_params, label):
    """
    Run a weighted least-squares fit and return a result dictionary.

    Parameters
    ----------
    model_func : callable   — f(r, *params)
    p0         : list       — initial parameter guess
    bounds     : tuple      — (lower_bounds, upper_bounds)
    n_params   : int        — number of free parameters (for chi2_red dof)
    label      : str        — human-readable model name for printing

    Returns
    -------
    dict with keys: ok, popt, perr, chi2_red
    """
    try:
        popt, pcov = curve_fit(
            model_func,
            r_data,
            N_data,
            p0=p0,
            sigma=N_err,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=20000,
        )
        perr = np.sqrt(np.diag(pcov))
        resid = N_data - model_func(r_data, *popt)
        chi2_red = float(np.sum((resid / N_err) ** 2) / (len(N_data) - n_params))
        print(f"\n{label}:")
        for name, val, err in zip([f"p{i}" for i in range(n_params)], popt, perr):
            print(f"   {name} = {val:.5g} +/- {err:.5g}")
        print(f"   chi2_red = {chi2_red:.3f}  (dof = {len(N_data) - n_params})")
        return dict(ok=True, popt=popt, perr=perr, chi2_red=chi2_red)
    except (RuntimeError, ValueError) as exc:
        print(f"WARNING: {label} fit failed: {exc}")
        return dict(ok=False, popt=None, perr=None, chi2_red=np.nan)


# ── M1: cos fit ───────────────────────────────────────────────────────────────
res1 = run_fit(
    model_cos,
    p0=[N_max, 1.0 / 300.0],
    bounds=([0, 0], [np.inf, np.inf]),
    n_params=2,
    label="M1  N = A*cos(B*r)",
)

# ── M2: cos^2 fit ─────────────────────────────────────────────────────────────
res2 = run_fit(
    model_cos2,
    p0=[N_max, 1.0 / 300.0],
    bounds=([0, 0], [np.inf, np.inf]),
    n_params=2,
    label="M2  N = A*cos^2(B*r)",
)

# ── M3: full 5-parameter fit ──────────────────────────────────────────────────
# Initial guesses:
#   N0    ~ N_max
#   K     ~ 1/300  (same scale as B above)
#   beta  ~ 1      (start with cosine^1, let it float)
#   r0    ~ R_FIT_MAX / 2  (vignetting scale in the middle of the fit range)
#   gamma ~ 0.5    (mild power-law slope)
res3 = run_fit(
    model_full,
    p0=[N_max, 1.0 / 300.0, 1.0, R_FIT_MAX / 2.0, 0.5],
    bounds=([0, 0, 0, 1, 0], [np.inf, np.inf, 10, np.inf, 10]),
    n_params=5,
    label="M3  N = N0*cos(K*r)^beta * (1+(r/r0)^2)^(-gamma)",
)

In [ ]:
# ── Overlay plot: data + three fit curves ────────────────────────────────────
#
# Visual convention
#   Data inner (r <= R_FIT_MAX) : filled circles, viridis colour scale
#   Data outer (r >  R_FIT_MAX) : hollow circles, same colour scale
#   Fit curves on [0, R_FIT_MAX]: solid lines
#   Extrapolation beyond        : dash-dot lines (same colour)
#   Vertical boundary           : grey dotted vertical line
#
# Colour scheme for the three models
M_COLORS = {"M1": "tomato", "M2": "royalblue", "M3": "forestgreen"}

r_max_plot = geom_fit_all["r_mm"].max() * 1.05
r_all = np.linspace(0, r_max_plot, 600)
r_solid = r_all[r_all <= R_FIT_MAX]
r_dash = r_all[(r_all >= R_FIT_MAX) & (r_all <= R_FITPLOT_MAX)]

fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)

# ── Data: inner (filled) ─────────────────────────────────────────────────────
vmin_c = geom_fit_all["n_alerts"].min()
vmax_c = geom_fit_all["n_alerts"].max()

sc_in = ax.scatter(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    c=geom_inner["n_alerts"],
    cmap="jet",
    vmin=vmin_c,
    vmax=vmax_c,
    s=35,
    zorder=4,
    label=rf"CCD  $r \leq {R_FIT_MAX:.0f}$ mm  (fit range)",
)
ax.errorbar(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    yerr=geom_inner["n_err"],
    fmt="none",
    ecolor="steelblue",
    elinewidth=0.8,
    capsize=2,
    alpha=0.55,
    zorder=3,
)

# ── Data: outer (hollow) ─────────────────────────────────────────────────────
if len(geom_outer) > 0:
    ax.scatter(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        c=geom_outer["n_alerts"],
        cmap="jet",
        vmin=vmin_c,
        vmax=vmax_c,
        s=35,
        zorder=4,
        marker="o",
        facecolors="none",
        linewidths=1.2,
        label=rf"CCD  $r > {R_FIT_MAX:.0f}$ mm  (excluded from fits)",
    )
    ax.errorbar(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        yerr=geom_outer["n_err"],
        fmt="none",
        ecolor="steelblue",
        elinewidth=0.8,
        capsize=2,
        alpha=0.35,
        zorder=3,
    )

# ── M1: cos fit ───────────────────────────────────────────────────────────────
if res1["ok"]:
    A1, B1 = res1["popt"]
    dA1, dB1 = res1["perr"]
    f1 = 1.0 / B1
    df1 = f1**2 * dB1
    lbl1 = (
        rf"M1: $A\cos(Br)$  $\chi^2_{{\rm red}}={res1['chi2_red']:.2f}$"
        f"\n$A={A1:.0f}\\pm{dA1:.0f}$,  "
        f"$B={B1:.5f}\\pm{dB1:.5f}$ rad/mm"
        f"\n$f_{{\\rm eff}}=1/B={f1:.0f}\\pm{df1:.0f}$ mm"
    )
    ax.plot(r_solid, model_cos(r_solid, A1, B1), color=M_COLORS["M1"], lw=2.0, ls="-", label=lbl1, zorder=5)
    ax.plot(
        r_dash,
        model_cos(r_dash, A1, B1),
        color=M_COLORS["M1"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M1 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

# ── M2: cos^2 fit ─────────────────────────────────────────────────────────────
if res2["ok"]:
    A2, B2 = res2["popt"]
    dA2, dB2 = res2["perr"]
    f2 = 1.0 / B2
    df2 = f2**2 * dB2
    lbl2 = (
        rf"M2: $A\cos^2(Br)$  $\chi^2_{{\rm red}}={res2['chi2_red']:.2f}$"
        f"\n$A={A2:.0f}\\pm{dA2:.0f}$,  "
        f"$B={B2:.5f}\\pm{dB2:.5f}$ rad/mm"
        f"\n$f_{{\\rm eff}}=1/B={f2:.0f}\\pm{df2:.0f}$ mm"
    )
    ax.plot(r_solid, model_cos2(r_solid, A2, B2), color=M_COLORS["M2"], lw=2.0, ls="-", label=lbl2, zorder=5)
    ax.plot(
        r_dash,
        model_cos2(r_dash, A2, B2),
        color=M_COLORS["M2"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M2 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

# ── M3: full 5-parameter fit ──────────────────────────────────────────────────
if res3["ok"]:
    N3, K3, beta3, r03, gam3 = res3["popt"]
    dN3, dK3, db3, dr03, dgam3 = res3["perr"]
    lbl3 = (
        rf"M3: $N_0\cos(Kr)^\beta(1+r^2/r_0^2)^{{-\gamma}}$  "
        rf"$\chi^2_{{\rm red}}={res3['chi2_red']:.2f}$"
        f"\n$N_0={N3:.0f}\\pm{dN3:.0f}$,  "
        f"$K={K3:.5f}\\pm{dK3:.5f}$ rad/mm"
        f"\n$\\beta={beta3:.2f}\\pm{db3:.2f}$,  "
        f"$r_0={r03:.1f}\\pm{dr03:.1f}$ mm,  "
        f"$\\gamma={gam3:.2f}\\pm{dgam3:.2f}$"
    )
    ax.plot(
        r_solid,
        model_full(r_solid, *res3["popt"]),
        color=M_COLORS["M3"],
        lw=2.0,
        ls="-",
        label=lbl3,
        zorder=5,
    )
    ax.plot(
        r_dash,
        model_full(r_dash, *res3["popt"]),
        color=M_COLORS["M3"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M3 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

# ── Fit boundary ──────────────────────────────────────────────────────────────
ax.axvline(R_FIT_MAX, color="gray", lw=1.0, ls=":", label=rf"Fit boundary $r={R_FIT_MAX:.0f}$ mm")

# ── Colour bar ────────────────────────────────────────────────────────────────
cbar = fig.colorbar(sc_in, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("N alerts per CCD", fontsize=8)

ax.set_xlabel(
    r"Radial distance from camera centre  $r = \sqrt{x^2+y^2}$  [mm]",
    fontsize=10,
)
ax.set_ylabel("Number of alerts per CCD  $N$", fontsize=10)
ax.set_title(
    rf"Alert count vs. focal-plane radius — three models, fit range $r \leq {R_FIT_MAX:.0f}$ mm",
    fontsize=11,
)
ax.set_xlim(-0.5, r_max_plot)
ax.set_ylim(bottom=0)
ax.legend(fontsize=7, loc="upper right", ncol=2)

savefig("alert_count_vs_radius_3models")
plt.show()

In [ ]:
# ── Model comparison table ───────────────────────────────────────────────────
# Collect reduced chi-squared and AIC (Akaike Information Criterion) for each
# model to rank their goodness-of-fit on the inner data.
#
# AIC = chi2 + 2 * k   where  chi2 = sum((resid/sigma)^2), k = number of params
# Lower AIC = better trade-off between fit quality and model complexity.

models = [
    ("M1", "A*cos(B*r)", 2, model_cos, res1),
    ("M2", "A*cos^2(B*r)", 2, model_cos2, res2),
    ("M3", "N0*cos(K*r)^beta*(1+(r/r0)^2)^-gam", 5, model_full, res3),
]

rows = []
for mname, formula, k, func, res in models:
    if res["ok"]:
        resid = N_data - func(r_data, *res["popt"])
        chi2_tot = float(np.sum((resid / N_err) ** 2))
        aic = chi2_tot + 2 * k
        rows.append(
            dict(
                model=mname,
                formula=formula,
                n_params=k,
                chi2_red=round(res["chi2_red"], 3),
                AIC=round(aic, 1),
            )
        )
    else:
        rows.append(
            dict(
                model=mname,
                formula=formula,
                n_params=k,
                chi2_red=np.nan,
                AIC=np.nan,
            )
        )

df_cmp = pd.DataFrame(rows).set_index("model")
print(f"\nModel comparison (fit range r <= {R_FIT_MAX:.0f} mm, {len(r_data)} CCDs):")
display(df_cmp)

In [ ]:
# ── Residuals plot — one panel per model, side by side ───────────────────────
# Pull = (N_obs - N_fit) / sqrt(N_obs)  for the fit range only.

active = [
    (n, f, func, res)
    for n, f, func, res in [
        ("M1", model_cos, model_cos, res1),
        ("M2", model_cos2, model_cos2, res2),
        ("M3", model_full, model_full, res3),
    ]
    if res["ok"]
]

# Re-build with correct tuples
active = []
for mname, formula, k, func, res in models:
    if res["ok"]:
        active.append((mname, func, res))

if active:
    n_mod = len(active)
    fig_r, axes_r = plt.subplots(
        2,
        n_mod,
        figsize=(5 * n_mod, 8),
        constrained_layout=True,
    )
    # Ensure axes_r is always 2-D
    if n_mod == 1:
        axes_r = axes_r.reshape(2, 1)

    for col, (mname, func, res) in enumerate(active):
        N_pred = func(r_data, *res["popt"])
        pull = (N_data - N_pred) / N_err
        color = M_COLORS[mname]

        # Row 0: pull vs r
        axes_r[0, col].scatter(
            r_data,
            pull,
            c=pull,
            cmap="RdBu_r",
            vmin=-3,
            vmax=3,
            s=28,
            zorder=3,
        )
        axes_r[0, col].axhline(0, color="k", lw=0.8, ls="--")
        axes_r[0, col].axhline(+2, color="gray", lw=0.6, ls=":")
        axes_r[0, col].axhline(-2, color="gray", lw=0.6, ls=":")
        axes_r[0, col].set_xlabel(r"$r$ [mm]", fontsize=9)
        axes_r[0, col].set_ylabel(
            r"$(N_{\rm obs}-N_{\rm fit})/\sqrt{N_{\rm obs}}$",
            fontsize=8,
        )
        axes_r[0, col].set_title(
            rf"{mname} residuals  $\chi^2_{{\rm red}}={res['chi2_red']:.2f}$",
            fontsize=9,
            color=color,
        )

        # Row 1: pull histogram
        axes_r[1, col].hist(
            pull,
            bins=20,
            color=color,
            alpha=0.75,
            edgecolor="k",
            linewidth=0.5,
        )
        axes_r[1, col].axvline(0, color="k", lw=1)
        axes_r[1, col].axvline(+2, color="gray", lw=0.8, ls=":")
        axes_r[1, col].axvline(-2, color="gray", lw=0.8, ls=":")
        axes_r[1, col].set_xlabel(
            r"$(N_{\rm obs}-N_{\rm fit})/\sqrt{N_{\rm obs}}$",
            fontsize=8,
        )
        axes_r[1, col].set_ylabel("Number of CCDs", fontsize=8)
        axes_r[1, col].set_title(
            f"mean={np.mean(pull):.2f},  std={np.std(pull):.2f}",
            fontsize=9,
        )

    fig_r.suptitle(
        rf"Residuals — fit range $r \leq {R_FIT_MAX:.0f}$ mm  ({len(r_data)} CCDs)",
        fontsize=11,
    )
    savefig("alert_count_vs_radius_residuals_3models")
    plt.show()
else:
    print("No model converged — residuals plot skipped.")